# Gold - Deteccion de fraude

Parte 4 del proyecto: leer Silver, aplicar reglas de riesgo, generar `fraud_alerts` y construir una tabla tabular para analisis de relaciones.

In [23]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("gold-fraud-job")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "admin123")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.gold")
print("Spark e Iceberg listos para Gold")

Spark e Iceberg listos para Gold


## 1) Leer Silver y preparar base

In [24]:
from pyspark.sql import functions as F

silver_df = spark.table("lakehouse.silver.payments_enriched")

base_df = silver_df.select(
    "payment_id",
    "event_ts",
    "customer_id",
    "card_id",
    "merchant_id",
    "device_id",
    "ip",
    "country",
    "amount",
    "currency",
    "status",
    "tx_count_card_5m",
    "distinct_merchants_card_10m",
    "distinct_countries_card_1h",
    "distinct_cards_device_1h",
    "declined_ratio_card_1h"
)

base_df.show(5, truncate=False)

+------------------------------------+--------------------------+-----------+----------+-----------+---------+---------------+-------+------+--------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|payment_id                          |event_ts                  |customer_id|card_id   |merchant_id|device_id|ip             |country|amount|currency|status  |tx_count_card_5m|distinct_merchants_card_10m|distinct_countries_card_1h|distinct_cards_device_1h|declined_ratio_card_1h|
+------------------------------------+--------------------------+-----------+----------+-----------+---------+---------------+-------+------+--------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|39f23bd3-3cc4-4917-b77b-5b0de33cb4bd|2026-03-25 12:24:32.93659 |CUST_00049 |CARD_00003|MERCH_00075|DEV_00028|108.31.59.18   |ET     |193.84|GBP     |approved|1

## 2) Reglas de fraude, reasons y risk_score

In [25]:
ALERT_THRESHOLD = 65

rules_df = (
    silver_df
    .withColumn("rule_high_tx_velocity",     F.col("tx_count_card_5m") >= 12)
    .withColumn("rule_many_merchants",       F.col("distinct_merchants_card_10m") >= 12)
    .withColumn("rule_many_countries",       F.col("distinct_countries_card_1h") >= 8)
    .withColumn("rule_device_shared",        F.col("distinct_cards_device_1h") >= 16)
    .withColumn("rule_high_declined_ratio",  F.col("declined_ratio_card_1h") >= 0.65)
    .withColumn("rule_high_amount",          F.col("amount") >= 700.00)
)

scored_df = (
    rules_df
    .withColumn(
        "risk_score_raw",
        F.when(F.col("rule_high_tx_velocity"),    F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_many_merchants"),      F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_many_countries"),      F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_device_shared"),       F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_high_declined_ratio"), F.lit(25)).otherwise(F.lit(0)) +
        F.when(F.col("rule_high_amount"),         F.lit(15)).otherwise(F.lit(0))
    )
    .withColumn("risk_score", F.least(F.lit(100), F.col("risk_score_raw")).cast("int"))
    .withColumn(
        "is_fraud_alert",
        F.col("risk_score") >= ALERT_THRESHOLD
    )
    .withColumn(
        "reasons",
        F.expr("""
            filter(
                array(
                    IF(rule_high_tx_velocity,    'high_tx_velocity_5m', NULL),
                    IF(rule_many_merchants,      'many_merchants_10m', NULL),
                    IF(rule_many_countries,      'many_countries_1h', NULL),
                    IF(rule_device_shared,       'device_shared_1h', NULL),
                    IF(rule_high_declined_ratio, 'high_declined_ratio_1h', NULL),
                    IF(rule_high_amount,         'high_amount', NULL)
                ),
                x -> x IS NOT NULL
            )
        """)
    )
)


## 3) Tabla gold.fraud_alerts

In [26]:
fraud_alerts_df = (
    scored_df
    .filter(F.col("risk_score") >= ALERT_THRESHOLD)
    .select(
        "payment_id",
        "event_ts",
        "customer_id",
        "card_id",
        "merchant_id",
        "device_id",
        "ip",
        "country",
        "amount",
        "currency",
        "status",
        "tx_count_card_5m",
        "distinct_merchants_card_10m",
        "distinct_countries_card_1h",
        "distinct_cards_device_1h",
        "declined_ratio_card_1h",
        "risk_score",
        "reasons"
    )
)

(
    fraud_alerts_df.writeTo("lakehouse.gold.fraud_alerts")
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print("Tabla Gold creada: lakehouse.gold.fraud_alerts")

Tabla Gold creada: lakehouse.gold.fraud_alerts


## 4) Tabla tabular para analisis de relaciones

In [27]:
relations_df = (
    scored_df
    .select(
        "payment_id",
        "event_ts",
        "customer_id",
        "card_id",
        "merchant_id",
        "device_id",
        "ip",
        "country",
        "amount",
        "currency",
        "status",
        "risk_score",
        "reasons"
    )
    .withColumn("is_alert", F.col("risk_score") >= ALERT_THRESHOLD)
)

(
    relations_df.writeTo("lakehouse.gold.payment_relations")
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print("Tabla Gold creada: lakehouse.gold.payment_relations")

Tabla Gold creada: lakehouse.gold.payment_relations


## 5) Validaciones rapidas

In [28]:
spark.sql("SELECT COUNT(*) AS total_alerts FROM lakehouse.gold.fraud_alerts").show()
spark.sql("SELECT COUNT(*) AS total_relations FROM lakehouse.gold.payment_relations").show()

spark.sql("""
SELECT payment_id, event_ts, card_id, merchant_id, risk_score, reasons
FROM lakehouse.gold.fraud_alerts
ORDER BY risk_score DESC, event_ts DESC
LIMIT 10
""").show(truncate=False)

+------------+
|total_alerts|
+------------+
|        6296|
+------------+

+---------------+
|total_relations|
+---------------+
|          50559|
+---------------+

+--------------------------------------------+--------------------------+----------+-----------+----------+-------------------------------------------------------------------------------------------+
|payment_id                                  |event_ts                  |card_id   |merchant_id|risk_score|reasons                                                                                    |
+--------------------------------------------+--------------------------+----------+-----------+----------+-------------------------------------------------------------------------------------------+
|c0dbe75a-43c0-481a-b00a-6de464e32c5e        |2026-03-25 14:45:32.390974|CARD_00052|MERCH_00015|95        |[high_tx_velocity_5m, many_merchants_10m, many_countries_1h, device_shared_1h, high_amount]|
|591d316f-7003-49df-b496-0bf00f5b

In [29]:
spark.sql("""
SELECT
  COUNT(*) AS total_alerts,
  MAX(risk_score) AS max_risk_score,
  ROUND(AVG(risk_score), 2) AS avg_risk_score
FROM lakehouse.gold.fraud_alerts
""").show()

spark.sql("""
SELECT COUNT(*) AS total_relations
FROM lakehouse.gold.payment_relations
""").show()

+------------+--------------+--------------+
|total_alerts|max_risk_score|avg_risk_score|
+------------+--------------+--------------+
|        6296|            95|         79.97|
+------------+--------------+--------------+

+---------------+
|total_relations|
+---------------+
|          50559|
+---------------+

